# Visualization of contact maps in *M. racemosus* using cooltools tutorial

## Import libraries

In [ ]:
# import standard python libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
# import cooltools and cooler
import cooltools
import cooler

In [ ]:
# Import libraries for plotting
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cooltools.lib.plotting

from matplotlib.ticker import EngFormatter # For adding bp to plots
import matplotlib.patches as patches # For adding circles to the plots

## Set format for file

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'

## Check available resolutions in mcool file

In [ ]:
# Resolutions in mcool file
cooler.fileops.list_coolers('/Users/emma/Documents/NMBU/Master/Master/gzMucFlav1/nf-core:HiC pipeline files/gzMucRace1.mcool')

## Import files

In [ ]:
# Load the cool file at 320 kb resolution
res_320k = cooler.Cooler('/Users/emma/Documents/NMBU/Master/Master/gzMucFlav1/nf-core:HiC pipeline files/gzMucRace1.mcool::resolutions/320000') 
# The resolution of contact maps
resolution = res_320k.binsize // 1000 
print(resolution)

## Inspect the mcool file

In [ ]:
# to print chromosome names and binsize for this cooler
print(f'chromosomes: {res_320k.chromnames}, binsize: {res_320k.binsize}')

In [ ]:
# Makes a list of chromosomes and binsize present in this cooler file. 
chromstarts = []
for i in res_320k.chromnames:
    print(f'{i} : {res_320k.extent(i)}')
    chromstarts.append(res_320k.extent(i)[0])

## Filtering out scaffolds

In [ ]:
# Filtering out scaffolds smaller than 10 000
min_size = 100_000

chroms_to_keep = [c for c in res_320k.chromnames if res_320k.chromsizes[c] >= min_size]

print("Chromosomes kept:", chroms_to_keep) # Print the chromosomes that we plot. 

In [ ]:
# Get the bins for the chromosomes we keep
bins_res_320k = res_320k.bins()[:]
keep_mask_res320k = bins_res_320k['chrom'].isin(chroms_to_keep)

In [ ]:
# Fetch full genome-wide matrix (
full_matrix_res320k = res_320k.matrix(balance=False)[:]

# Subset matrix to only the filtered chromosomes
matrix_filtered_res320k = full_matrix_res320k[keep_mask_res320k.values, :][:, keep_mask_res320k.values]

In [ ]:
# Shorten the chromosome names so that the plot only shows the chromosome number
chromstarts = []
pos = 0
short_labels = []
for c in chroms_to_keep:
    n_bins = (bins_res_320k[bins_res_320k['chrom'] == c].shape[0])
    chromstarts.append(pos)
    short_labels.append(c.split('_')[-1])
    pos += n_bins


# Visualizing the Hi-C data

### Genome-wide plot of *M. racemosus*

In [ ]:
# Plot
f, ax = plt.subplots(figsize=(8, 8)) # size of the plot
norm = LogNorm(vmin=1, vmax=1000) # Changed this to make the colors clearer

im = ax.matshow(matrix_filtered_res320k, cmap='fall', norm=norm) # Plott filtered genome-wide contact map with fall colours

# Colorbar settings
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Contact counts (log scale)', fontsize=16)
cbar.ax.tick_params(labelsize=9)

# Set the x/y-axis number at the middle of the chromosome
chromcenters = []
for i in range(len(chromstarts)):
    if i < len(chromstarts) - 1:
        chromcenters.append((chromstarts[i] + chromstarts[i+1]) / 2)
    else:
        chromcenters.append((chromstarts[i] + pos) / 2)

# Add ticks and labels to the x- and y-axis
ax.set_xticks(chromcenters)
ax.set_yticks(chromcenters)
ax.set_xticklabels(short_labels, fontsize=11)
ax.set_yticklabels(short_labels, fontsize=11)
ax.xaxis.tick_bottom()
ax.set_title(f"M. racemosus (320 k)", fontsize=20)

ax.set_xlabel("Chromosome", fontsize=16, labelpad=10)
ax.set_ylabel("Chromosome", fontsize=16, labelpad=10)

# Adds grey lines at chromosome boundaries 
for s in chromstarts:
    ax.axhline(s - 0.5, color='grey', lw=0.5)
    ax.axvline(s - 0.5, color='grey', lw=0.5)


plt.tight_layout()
plt.show()


### Subgenome 1 plot of *M. racemosus*

In [ ]:
# Filtering so we only get chromosomes larger than 100 000 and present in subgenome 1. Due to poor coverage in subgenome 2
min_size = 100_000

chroms_subgenome1  = [
    c for c in res_320k.chromnames
    if c.startswith("subgenome1_") and res_320k.chromsizes[c] >= min_size
]
print("Chromosomes kept:", chroms_subgenome1)

In [ ]:
# Get indices of bins for the chromosomes to keep
bins_subgenome1 = res_320k.bins()[:]
keep_mask_subgenome1 = bins_subgenome1['chrom'].isin(chroms_subgenome1)

In [ ]:
# Short labels for ticks
chromstarts = []
pos = 0
short_labels_subgenome1 = []
for c in chroms_subgenome1:
    n_bins = (bins_subgenome1[bins_subgenome1['chrom'] == c].shape[0])
    chromstarts.append(pos)
    short_labels_subgenome1.append(c.split('_')[-1])
    pos += n_bins

In [ ]:
# Fetch full genome-wide matrix (balanced or raw)
full_matrix = res_320k.matrix(balance=False)[:]

# Subset matrix to only keep large chromosomes in subgenome 1
matrix_subgenome1 = full_matrix[keep_mask_subgenome1.values, :][:, keep_mask_subgenome1.values]

In [ ]:
# Plot
f, ax = plt.subplots(figsize=(8, 8))
norm = LogNorm(vmin=1, vmax=1000) # Changed this to make the colors clearer

im = ax.matshow(matrix_subgenome1, cmap='fall', norm=norm) # Plott filtered genome-wide contact map with fall colours

# colorbar settings
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Contact counts (log scale)', fontsize=16)
cbar.ax.tick_params(labelsize=12)

# Set the x/y-axis number at the middle of the chromosome
chromcenters = []
for i in range(len(chromstarts)):
    if i < len(chromstarts) - 1:
        chromcenters.append((chromstarts[i] + chromstarts[i+1]) / 2)
    else:
        chromcenters.append((chromstarts[i] + pos) / 2)

ax.set_xticks(chromcenters)
ax.set_yticks(chromcenters)
ax.set_xticklabels(short_labels_subgenome1, fontsize=12)
ax.set_yticklabels(short_labels_subgenome1, fontsize=12)
ax.xaxis.tick_bottom()
ax.set_title(f"M. racemosus (320 k)", fontsize=20) 

ax.set_xlabel("Chromosome", fontsize=16, labelpad=10)
ax.set_ylabel("Chromosome", fontsize=16, labelpad=10)

# Adds grey lines at chromosome boundaries 
for s in chromstarts:
    ax.axhline(s - 0.5, color='grey', lw=0.5)
    ax.axvline(s - 0.5, color='grey', lw=0.5)


plt.tight_layout()
plt.show()


## Functions used for plotting (found in the cooltools tutorial)

In [ ]:
# Function for adding basepairs to plots
bp_formatter = EngFormatter('b') # Here used to formate to get genomic coordinates in megabases
def format_ticks(ax, x=True, y=True, rotate=True): # To format axis ticks 
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_top()  # move x ticks to top for genome plots
    if rotate:
        ax.tick_params(axis='x')

## Predicted centromeres in *M. racemosus*

In [ ]:
centromeres = {
    "subgenome1_scaffold_2": (4713049, 5677333),
    "subgenome1_scaffold_3": (2619479, 16444124)
}

## For loop going through all chromosomes with predicted centromere

In [ ]:
# Loop over all chromosomes with predicted centromeres 
# Loop goes through all the annotated centromeres and plot the contact map for the given chromosome together with the centromere annotation.
for chrom, (cent_start, cent_end) in centromeres.items():

    # Full chromosome coordinates
    start = 0 # all chromosomes start with 0
    end = res_320k.chromsizes[chrom] # end is the chromosome size
    region = f"{chrom}:{start}-{end}" # plot the whole chromosome
    extents = (start, end, end, start)

    # Chromosome label from short_labels
    chrom_index = chroms_to_keep.index(chrom) # Uses chroms_to_keep to label chromosomes 
    chrom_label = short_labels[chrom_index] # Use short_labels to shorten the chromosomes name in chroms_to_keep

    # Fetch matrix for the region (the whole chromosome)
    matrix = res_320k.matrix(balance=False).fetch(region)

    # Plot
    f, ax = plt.subplots(figsize=(8, 8)) # plot size
    norm = LogNorm(vmin=1, vmax=1000)
    im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Contact counts (log scale)', fontsize=14)

    # Axis labels
    ax.set_xlabel('Genomic position', fontsize = 14)
    ax.set_ylabel('Genomic position', fontsize = 14)

    # Use format_ticks to get basepairs along the x- and y-axis
    format_ticks(ax)

   # Draw a circle around the centromere 
    cent_mid = (cent_start + cent_end) / 2 # Find the middle of the centromere interval
    radius = 0.03 * (end - start)  # The radius of the circle is 3% of chromosome length
    circle = patches.Circle((cent_mid, cent_mid), radius=radius, # Draw a black circle around the centromere
                            edgecolor='black', facecolor='none', lw=2)
    ax.add_patch(circle) # Add the circle to the plot

    # Plot Title (Chromosome number and resolution)
    ax.set_title(f'Chromosome {chrom_label} in M. racemosus ({resolution} k)', fontsize = 16)

    plt.tight_layout()
    plt.show()


## Loop through all chromosomes in subgenome 1

In [ ]:
# Loop over all chromosomes in subgenome 1 regardless of centromere prediction
for chrom in chroms_subgenome1:
    # Full chromosome coordinates
    start = 0
    end = res_320k.chromsizes[chrom]
    region = f"{chrom}:{start}-{end}"
    extents = (start, end, end, start)

    # Get short chromosome label
    chrom_index = chroms_subgenome1.index(chrom)
    chrom_label = short_labels[chrom_index]

    # Fetch Hi-C matrix
    matrix = res_320k.matrix(balance=False).fetch(region)

    # Plot
    f, ax = plt.subplots(figsize=(8, 8))
    norm = LogNorm(vmin=1, vmax=1000)
    im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Contact counts (log scale)', fontsize=14)

    # Axis labels
    ax.set_xlabel('Genomic position', fontsize = 14)
    ax.set_ylabel('Genomic position', fontsize = 14)

    format_ticks(ax)

    # Title
    ax.set_title(f'Chromosome {chrom_label} in M. racemosus ({resolution} k)', fontsize = 16)

    plt.tight_layout()
    plt.show()



### Tested at 160 kb resolution

In [ ]:
# Load the cool file at a specific resolution
res_160k = cooler.Cooler('/Users/emma/Documents/NMBU/Master/Master/gzMucFlav1/nf-core:HiC pipeline files/gzMucRace1.mcool::resolutions/160000') 
# The resolution of contact maps
resolution_160k = res_160k.binsize // 1000 
print(resolution_160k)

In [ ]:
# Loop over all chromosomes in subgenome 1 regardless of centromere prediction
for chrom in chroms_subgenome1:
    # Full chromosome coordinates
    start = 0
    end = res_160k.chromsizes[chrom]
    region = f"{chrom}:{start}-{end}"
    extents = (start, end, end, start)

    # Get short chromosome label
    chrom_index = chroms_subgenome1.index(chrom)
    chrom_label = short_labels[chrom_index]

    # Fetch Hi-C matrix
    matrix = res_160k.matrix(balance=False).fetch(region)

    # Plot
    f, ax = plt.subplots(figsize=(8, 8))
    norm = LogNorm(vmin=1, vmax=1000)
    im = ax.matshow(matrix, cmap='fall', norm=norm, extent=extents)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Contact counts (log scale)', fontsize=14)

    # Axis labels
    ax.set_xlabel('Genomic position', fontsize = 14)
    ax.set_ylabel('Genomic position', fontsize = 14)

    format_ticks(ax)

    # Title
    ax.set_title(f'Chromosome {chrom_label} in M. racemosus ({resolution_160k} k)', fontsize = 16)

    plt.tight_layout()
    plt.show()

